![Universidad Espíritu Santo](https://raw.githubusercontent.com/andresrubiop/miar0525-estudiantes/main/utils/logo-uees-color.png)

<div style="background:#821436;color:#FFFFFF;padding:14px 18px;border-radius:10px;margin:6px 0 12px 0"><div style="font-size:12px;letter-spacing:.08em;text-transform:uppercase;opacity:.9">Aprendizaje Automático · MIAR0525 · Semana 2 · Notebook del estudiante · no calificable</div><div style="font-size:22px;font-weight:700;margin-top:4px">E2.1 · Regresión lineal y regularización</div><div style="font-size:12px;opacity:.9;margin-top:4px">Postgrado · Maestría en Inteligencia Artificial · UEES</div></div>

| | |
|---|---|
| **Objetivo** | Entender la regresión lineal por dentro (ecuación normal y descenso de gradiente) y usar la regularización Ridge y Lasso para controlar la complejidad. |
| **Resultado de aprendizaje** | RDA1 · competencias CG-G1 y CE-G1 |
| **Duración** | ≈ 4 h |
| **Teoría** | Manual M2 §1–2 · Animaciones A2.1 y A2.2 · Video V2.1 |
| **Datos** | Sintéticos · **California Housing** (`fetch_california_housing`: 20 640 distritos, 8 variables, valor mediano de la vivienda) |

**Niveles:** 1 · ecuación normal y gradiente desde cero → 2 · scikit-learn → 3 · California Housing con regularización → 4 · reto.

## 0 · Configuración

In [ ]:
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import sklearn
from cycler import cycler

warnings.filterwarnings("ignore", category=FutureWarning)
SEED = 2026
UEES = {"vino": "#821436", "azul": "#1F6F8B", "ocre": "#C28E0E", "verde": "#3A7D44", "gris": "#77787B"}
plt.rcParams.update({
    "axes.prop_cycle": cycler(color=list(UEES.values())), "axes.titlecolor": UEES["vino"],
    "axes.titleweight": "bold", "axes.edgecolor": UEES["gris"], "axes.grid": True, "grid.color": "#EEE8EA",
    "axes.spines.top": False, "axes.spines.right": False, "figure.dpi": 110, "legend.frameon": False,
})
print(f"scikit-learn {sklearn.__version__} · semilla {SEED}")

## Nivel 1 · Desde cero (prelaboratorio)

Generamos 200 observaciones de $y = 3 + 2x_1 - 1.5x_2 + \varepsilon$ con $\varepsilon \sim N(0, 1)$. La variable $x_2$ está medida en una escala 50 veces mayor que $x_1$, como pasa con datos reales (metros frente a kilómetros, dólares frente a miles de dólares).

In [ ]:
rng = np.random.default_rng(SEED)
n = 200
x1 = rng.normal(0, 1, n)
x2 = rng.normal(0, 50, n)
y = 3 + 2 * x1 - 1.5 * (x2 / 50) + rng.normal(0, 1, n)
X = np.column_stack([x1, x2])
print(f"escala de x1: DE {x1.std():.2f} · escala de x2: DE {x2.std():.2f}")

### 1.1 Ecuación normal
Con una columna de unos para el intercepto, la solución de mínimos cuadrados es $\hat\beta = (X^\top X)^{-1}X^\top y$. En la práctica **no se invierte la matriz**: se resuelve el sistema $X^\top X\,\beta = X^\top y$.

In [ ]:
def agregar_intercepto(X):
    return np.column_stack([np.ones(len(X)), X])


def ecuacion_normal(X, y):
    """Devuelve [b0, b1, ..., bp] resolviendo X^T X beta = X^T y (X sin la columna de unos)."""
    # TODO: agrega el intercepto y resuelve el sistema con np.linalg.solve.
    Xb = ...
    return ...


beta_ne = ecuacion_normal(X, y)
print("Ecuación normal:", np.round(beta_ne, 4))

### 1.2 Descenso de gradiente
La pérdida es el ECM, $L(\beta)=\frac1n\|X\beta-y\|^2$, y su gradiente $\nabla L = \frac2n X^\top(X\beta - y)$. Cada paso: $\beta \leftarrow \beta - \eta\,\nabla L$.

In [ ]:
def descenso_gradiente(X, y, eta, iters=500):
    """Devuelve (beta, historial de ECM). X sin la columna de unos."""
    Xb = agregar_intercepto(X)
    beta = np.zeros(Xb.shape[1])
    hist = []
    for _ in range(iters):
        # TODO: calcula el residuo, el gradiente y actualiza beta; guarda el ECM.
        resid = ...
        grad = ...
        beta = ...
        hist.append(...)
        if not np.isfinite(hist[-1]) or hist[-1] > 1e12:
            break
    return beta, np.array(hist)


# Sin escalar, la variable grande obliga a usar una tasa diminuta
for eta in (1e-4, 3e-4, 8e-4):
    b, h = descenso_gradiente(X, y, eta)
    estado = "diverge" if not np.isfinite(h[-1]) or h[-1] > 1e6 else f"ECM final {h[-1]:.3f}"
    print(f"sin escalar · η = {eta:g}: {estado}")

Con $x_2$ sin escalar, la tasa estable es minúscula y el intercepto casi no se mueve en 500 pasos. **Estandarizamos** (media 0, DE 1) y repetimos: ahora una tasa de 0.1 converge en pocas decenas de pasos.

In [ ]:
mu, sd = X.mean(axis=0), X.std(axis=0)
Xs = (X - mu) / sd

fig, ax = plt.subplots(figsize=(7.5, 3.6))
curvas = {}
for eta in (0.01, 0.1, 0.5, 1.05):
    b, h = descenso_gradiente(Xs, y, eta, iters=150)
    curvas[eta] = (b, h)
    ax.plot(np.clip(h, 0, 50), label=f"η = {eta}")
ax.axhline(np.mean((agregar_intercepto(X) @ beta_ne - y) ** 2), color="#1C1A1B", ls="--", lw=1, label="mínimo (ecuación normal)")
ax.set(title="Convergencia del descenso de gradiente (x estandarizada)", xlabel="iteración", ylabel="ECM", yscale="log")
ax.legend()
plt.show()

**Qué observar.** η = 0.01 es lenta, η = 0.1 y 0.5 convergen rápido y η = 1.05 **diverge**: con variables estandarizadas el límite es $2/\lambda_{\max}$ del hessiano $\frac2n X^\top X$, cercano a 1 (como en la animación A2.1).

Los coeficientes obtenidos con $x$ estandarizada se pueden llevar de vuelta a la escala original: $\beta_j = \beta^{(s)}_j/s_j$ y $\beta_0 = \beta^{(s)}_0 - \sum_j \beta^{(s)}_j\mu_j/s_j$.

In [ ]:
beta_gd_s = curvas[0.1][0]
beta_gd = np.concatenate([[beta_gd_s[0] - np.sum(beta_gd_s[1:] * mu / sd)], beta_gd_s[1:] / sd])
print("Gradiente (escala original):", np.round(beta_gd, 4))
assert np.allclose(beta_gd, beta_ne, atol=1e-3), "El descenso de gradiente no llegó a la solución de la ecuación normal."
print("✓ El gradiente converge a la ecuación normal")

### 1.3 Ridge desde cero
Ridge minimiza $\|y - X\beta\|^2 + \alpha\|\beta\|^2$ **sin penalizar el intercepto**. Si centramos $X$ e $y$, la solución es $\hat\beta = (X_c^\top X_c + \alpha I)^{-1} X_c^\top y_c$ y el intercepto es $\bar y - \bar x^\top\hat\beta$.

In [ ]:
def ridge_cerrado(X, y, alpha):
    """Devuelve (intercepto, coeficientes) de Ridge con la convención de scikit-learn."""
    # TODO: centra X e y, resuelve el sistema penalizado y recupera el intercepto.
    xm, ym = ...
    Xc, yc = ...
    coef = ...
    return ..., coef


for alpha in (0, 10, 100, 1000):
    b0, coef = ridge_cerrado(Xs, y, alpha)
    print(f"α = {alpha:>4}: intercepto {b0:.3f} · coeficientes {np.round(coef, 3)}")

**Qué observar.** Al subir α los coeficientes se encogen hacia 0, pero el intercepto (la media de $y$) no cambia.

## Nivel 2 · Con scikit-learn

In [ ]:
from sklearn.linear_model import Lasso, LinearRegression, Ridge, RidgeCV

lr = LinearRegression().fit(X, y)
print("LinearRegression:", np.round(np.r_[lr.intercept_, lr.coef_], 4))
assert np.allclose(np.r_[lr.intercept_, lr.coef_], beta_ne), "La ecuación normal no coincide con LinearRegression."

for alpha in (10, 100):
    rd = Ridge(alpha=alpha).fit(Xs, y)
    b0, coef = ridge_cerrado(Xs, y, alpha)
    assert np.allclose(rd.coef_, coef) and np.isclose(rd.intercept_, b0), "Tu Ridge no coincide con scikit-learn."
print("✓ Tu ecuación normal y tu Ridge coinciden con scikit-learn")

**Ojo con las convenciones de α.** `Ridge` usa $\|y-X\beta\|^2 + \alpha\|\beta\|^2$, mientras que `Lasso` usa $\frac{1}{2n}\|y-X\beta\|^2 + \alpha\|\beta\|_1$: un mismo α no significa lo mismo en ambos. Por eso α **siempre se elige por validación cruzada**, nunca se copia de otro modelo.

Agregamos 6 variables de puro ruido y comparamos:

In [ ]:
X_ruido = np.column_stack([Xs, rng.normal(0, 1, (n, 6))])
nombres = ["x1", "x2"] + [f"ruido{k}" for k in range(1, 7)]
coefs = pd.DataFrame({
    "MCO": LinearRegression().fit(X_ruido, y).coef_,
    "Ridge α=50": Ridge(alpha=50).fit(X_ruido, y).coef_,
    "Lasso α=0.1": Lasso(alpha=0.1).fit(X_ruido, y).coef_,
}, index=nombres).round(3)
coefs

In [ ]:
ridge_cv = RidgeCV(alphas=np.logspace(-3, 3, 50)).fit(X_ruido, y)
print(f"RidgeCV eligió α = {ridge_cv.alpha_:.3g}")
print("Lasso deja en cero:", [v for v, c in zip(nombres, coefs["Lasso α=0.1"]) if c == 0])

**Qué observar.** MCO asigna coeficientes pequeños pero distintos de cero al ruido; Ridge los encoge a todos; **Lasso anula los del ruido** y conserva x1 y x2 (animación A2.2).

## Nivel 3 · Datos reales: California Housing

Cada fila es un distrito censal de California (1990). El objetivo es el valor mediano de la vivienda en cientos de miles de dólares. Separamos la prueba al inicio y todo el preprocesamiento va dentro del pipeline.

In [ ]:
from sklearn.datasets import fetch_california_housing
from sklearn.linear_model import LassoCV, lasso_path
from sklearn.metrics import mean_absolute_error, root_mean_squared_error
from sklearn.model_selection import KFold, cross_validate, train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

cal = fetch_california_housing(as_frame=True)
Xc, yc = cal.data, cal.target
Xc_tr, Xc_te, yc_tr, yc_te = train_test_split(Xc, yc, test_size=0.2, random_state=SEED)
cv = KFold(n_splits=5, shuffle=True, random_state=SEED)
print(f"{len(Xc_tr)} distritos de entrenamiento · {len(Xc_te)} de prueba · variables: {list(Xc.columns)}")

modelos = {
    "Lineal (MCO)": make_pipeline(StandardScaler(), LinearRegression()),
    "RidgeCV": make_pipeline(StandardScaler(), RidgeCV(alphas=np.logspace(-3, 3, 30))),
    "LassoCV": make_pipeline(StandardScaler(), LassoCV(alphas=np.logspace(-4, 0, 30), cv=5, random_state=SEED)),
}
filas = {}
for nombre, m in modelos.items():
    r = cross_validate(m, Xc_tr, yc_tr, cv=cv, scoring=("neg_mean_absolute_error", "neg_root_mean_squared_error"))
    filas[nombre] = {"MAE (CV)": f"{-r['test_neg_mean_absolute_error'].mean():.3f} ± {r['test_neg_mean_absolute_error'].std():.3f}",
                     "RMSE (CV)": f"{-r['test_neg_root_mean_squared_error'].mean():.3f} ± {r['test_neg_root_mean_squared_error'].std():.3f}"}
pd.DataFrame(filas).T

Con 8 variables y 16 mil filas, la regularización casi no cambia el error: **hay muchos más datos que parámetros** y el modelo lineal tiene sesgo, no varianza. La ruta de coeficientes muestra, aun así, cómo Lasso ordena las variables por importancia.

In [ ]:
Xs_tr = StandardScaler().fit_transform(Xc_tr)
alphas, rutas, _ = lasso_path(Xs_tr, yc_tr - yc_tr.mean(), alphas=np.logspace(-4, 0, 60))
fig, ax = plt.subplots(figsize=(8, 4))
for j, nombre in enumerate(Xc.columns):
    ax.plot(alphas, rutas[j], label=nombre)
ax.set(xscale="log", title="Ruta de coeficientes de Lasso (variables estandarizadas)", xlabel="α", ylabel="coeficiente")
ax.axhline(0, color="#1C1A1B", lw=0.8)
ax.legend(ncol=2, fontsize=8)
plt.show()
orden_salida = [Xc.columns[j] for j in np.argsort([alphas[np.nonzero(rutas[j])[0][0]] if np.any(rutas[j]) else 0 for j in range(8)])[::-1]]
print("Orden de entrada en la ruta (de más a menos importante):", orden_salida)

### 3.1 Residuos por región y por nivel de precio
Un error promedio bajo puede esconder errores **sistemáticos**. Dividimos por latitud (sur: Los Ángeles y San Diego, lat < 34.5; centro; norte: bahía de San Francisco y más al norte, lat ≥ 37.5) y por quintil del valor real.

In [ ]:
final = modelos["RidgeCV"].fit(Xc_tr, yc_tr)
pred_te = final.predict(Xc_te)
res = yc_te - pred_te
region = pd.cut(Xc_te["Latitude"], [0, 34.5, 37.5, 90], labels=["sur", "centro", "norte"])
tabla_region = pd.DataFrame({"residuo": res, "abs": res.abs(), "región": region}).groupby("región", observed=True).agg(
    n=("residuo", "size"), sesgo_medio=("residuo", "mean"), MAE=("abs", "mean")).round(3)
nivel = pd.qcut(yc_te, 5, labels=["Q1 (más baratos)", "Q2", "Q3", "Q4", "Q5 (más caros)"])
tabla_nivel = pd.DataFrame({"residuo": res, "nivel": nivel}).groupby("nivel", observed=True)["residuo"].mean().round(3)
print(f"Prueba · MAE {mean_absolute_error(yc_te, pred_te):.3f} · RMSE {root_mean_squared_error(yc_te, pred_te):.3f}")
print("Residuo medio (y − ŷ) por quintil del valor real:", tabla_nivel.to_dict())
tabla_region

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10.5, 4))
sc = axes[0].scatter(Xc_te["Longitude"], Xc_te["Latitude"], c=np.clip(res, -2, 2), cmap="RdBu", s=4, vmin=-2, vmax=2)
axes[0].set(title="Residuos en el mapa (azul: subestima)", xlabel="longitud", ylabel="latitud")
fig.colorbar(sc, ax=axes[0], label="y − ŷ")
axes[1].scatter(pred_te, res, s=4, alpha=0.4, color=UEES["azul"])
axes[1].axhline(0, color=UEES["vino"])
axes[1].set(title="Residuos frente a la predicción", xlabel="ŷ (cientos de miles de USD)", ylabel="y − ŷ")
plt.tight_layout()
plt.show()

**Qué observar.** Los residuos no son ruido. Por zona, el modelo **sobreestima** el centro del estado, y en el mapa aparecen bolsones azules (subestima) en la costa de la bahía y de Los Ángeles. Por nivel de precio, **subestima los distritos caros y sobreestima los baratos**: la recta "comprime" los extremos. La franja diagonal del gráfico de la derecha aparece porque el valor está **topado en 5.0** en los datos originales. Un modelo lineal no captura bien la geografía; los árboles y bosques de E2.4 sí.

## Nivel 4 · Reto: variables polinómicas + Lasso
Con `PolynomialFeatures(degree=2)` pasamos de 8 a 44 variables (cuadrados e interacciones). ¿Cuántas sobreviven con Lasso?

In [ ]:
from sklearn.preprocessing import PolynomialFeatures

resultados = []
for alpha in (0.001, 0.005, 0.02, 0.05):
    poly = make_pipeline(StandardScaler(), PolynomialFeatures(degree=2, include_bias=False), StandardScaler(),
                         Lasso(alpha=alpha, max_iter=20000))
    r = cross_validate(poly, Xc_tr, yc_tr, cv=cv, scoring="neg_mean_absolute_error")
    poly.fit(Xc_tr, yc_tr)
    nz = int(np.sum(poly[-1].coef_ != 0))
    resultados.append({"α": alpha, "variables ≠ 0": f"{nz} de {poly[-1].coef_.size}", "MAE (CV)": round(-r["test_score"].mean(), 3)})
tabla_poly = pd.DataFrame(resultados)
tabla_poly

In [ ]:
poly = make_pipeline(StandardScaler(), PolynomialFeatures(degree=2, include_bias=False), StandardScaler(), Lasso(alpha=0.02, max_iter=20000)).fit(Xc_tr, yc_tr)
nombres_poly = poly[1].get_feature_names_out(Xc.columns)
sobreviven = pd.Series(poly[-1].coef_, index=nombres_poly)
sobreviven[sobreviven != 0].sort_values(key=np.abs, ascending=False).round(3).head(10)

**Qué observar.** Con α pequeño Lasso conserva casi todas las variables y el MAE mejora respecto del modelo lineal; al subir α quedan pocas, casi siempre `MedInc`, términos de latitud y longitud y su interacción: la geografía entra por la puerta de las interacciones.

## Autoverificación

In [ ]:
assert np.allclose(np.r_[lr.intercept_, lr.coef_], beta_ne)
assert np.allclose(beta_gd, beta_ne, atol=1e-3)
assert (coefs.loc[[f"ruido{k}" for k in range(1, 7)], "Lasso α=0.1"] == 0).sum() >= 4, "Lasso debería anular la mayoría de las variables de ruido."
assert tabla_nivel.iloc[-1] > 0 > tabla_nivel.iloc[0], "Revisa el signo de los residuos (y − ŷ): los caros deberían quedar subestimados."
print("✓ E2.1 completo")

## Lista de cotejo (autoevaluación)

- [ ] Tu ecuación normal y tu Ridge coinciden con scikit-learn.
- [ ] Viste converger y diverger el descenso de gradiente, y el efecto de escalar.
- [ ] Comparaste MCO, Ridge y Lasso con variables de ruido.
- [ ] Analizaste los residuos por región en California Housing.
- [ ] Probaste variables polinómicas con Lasso.

**Reflexión:** ¿por qué la regularización casi no ayudó en California Housing y sí con las variables de ruido?

**Cómo te prepara para la Tarea 2:** vas a comparar modelos con validación cruzada y a justificar la regularización con evidencia, no por costumbre.

## Desafío opcional con IA agéntica · Elastic Net y la estabilidad de los coeficientes

**Objetivo.** Comparar Ridge, Lasso y Elastic Net y medir qué tan estables son sus coeficientes.

**Prompt inicial.** Úsalo en la herramienta que prefieras (Claude Code, Codex, Gemini en Colab, ChatGPT…), con este notebook resuelto como contexto. Pide primero un plan y revisa cada paso antes de aprobarlo.

```text
En el notebook resuelto E2.1 (regresión lineal y regularización con California Housing), agrega una sección que ajuste ElasticNetCV dentro de un Pipeline con StandardScaler y compare Ridge, Lasso y Elastic Net por validación cruzada. Después, con 50 muestras bootstrap, grafica la dispersión de cada coeficiente para los tres modelos y explica en español qué variables son estables, cuáles cambian y qué implica eso para interpretarlas.
```

**Cómo verificar el resultado**

- Todo el preprocesamiento está dentro del pipeline.
- Los errores se reportan con media ± desviación.
- La interpretación no confunde un coeficiente con un efecto causal.

**Declara el uso de IA** (norma f del sílabo): herramienta, prompts relevantes, qué verificaste tú y qué corregiste. El desafío es opcional y no se califica; lo que cuenta es que puedas explicar cada decisión.